# 🤖 Agent IA — Extraction & Qualification B2B d'entreprises (Luxembourg)

**Rapport de stage — INPT × 3LM Solutions, Tunisie**
Réalisé par : **Mohamed EL HARCHALI** · Encadrante : **Saidi Souhir**

---

Ce notebook implémente un **pipeline multi-agents** capable de :

1. Comprendre une demande client en langage naturel
2. Retrouver l'entreprise correspondante au Luxembourg (registre officiel)
3. Vérifier si l'entreprise est **B2B**
4. Répondre à la question posée, en s'appuyant si besoin sur l'historique complet de l'entreprise

| Outil | Rôle |
|---|---|
| 🦙 **Ollama** (`gpt-oss:120b`) | Compréhension de la demande + réponse rapide |
| 🏢 **OpenCorporates API** | Registre officiel des entreprises (Luxembourg) |
| 🔍 **Google AI Mode** (via SerpApi) | Qualification B2B / B2C |
| ⚡ **NVIDIA Nemotron** | Analyse approfondie des statements, chunk par chunk |
| 📊 **LangSmith** | Traçage de chaque étape du pipeline |

> 🔐 **Sécurité** : toutes les clés d'API sont récupérées via le gestionnaire de secrets de Google Colab (`google.colab.userdata`) — **aucune clé n'est écrite en clair dans ce notebook**. Pensez à renseigner vos propres secrets dans l'onglet 🔑 *Secrets* de Colab avant l'exécution : `Open`, `OLLAMA`, `SERPAPI_API_KEY`, `NVIDIA`, `LANGSMITH`.

## 📦 1. Installation des dépendances

Exécuter cette cellule une seule fois par session Colab.

In [ ]:
!pip install -q langsmith openai ollama google-search-results beautifulsoup4 requests

## 🔑 2. Configuration & clients

Toutes les clés sont chargées **dynamiquement** depuis le gestionnaire de secrets de Colab — rien n'est codé en dur ici.

In [ ]:
from langsmith import traceable
import os
import json
import re
import requests
import serpapi
from concurrent.futures import ThreadPoolExecutor
from bs4 import BeautifulSoup
from google.colab import userdata
from openai import OpenAI
from ollama import Client

# --- LangSmith : traçage de toutes les étapes du pipeline ---
os.environ['LANGSMITH_TRACING'] = "true"
os.environ['LANGSMITH_ENDPOINT'] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_API_KEY'] = f"{userdata.get('LANGSMITH')}"
os.environ['LANGSMITH_PROJECT'] = "other_one"

# --- Clés API (récupérées depuis les secrets Colab, jamais en dur) ---
OPENCORPORATES_TOKEN = userdata.get('Open')
OLLAMA_API_KEY = userdata.get('OLLAMA')
MODEL_NAME = "gpt-oss:120b-cloud"
SERPAPI_KEY = userdata.get('SERPAPI_API_KEY')
NVIDIA_KEY = userdata.get('NVIDIA')

# --- Clients ---
ollama_client = Client(
    host="https://ollama.com",
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"}
)
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_KEY
)

NOT_FOUND_MARKER = "RIEN_TROUVE"


## 🧹 3. Fonctions utilitaires

- `clean_html` : nettoie les champs HTML renvoyés par OpenCorporates
- `clean_json_fences` : retire les balises ```json``` que les LLM ajoutent parfois autour de leur réponse

In [ ]:
def clean_html(obj):
    if isinstance(obj, dict):
        return {k: clean_html(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_html(v) for v in obj]
    elif isinstance(obj, str):
        if "<" in obj and ">" in obj:
            return BeautifulSoup(obj, "html.parser").get_text(" ", strip=True)
        return obj
    return obj


def clean_json_fences(raw):
    """Retire les ```json ... ``` que le LLM ajoute parfois autour du JSON."""
    return raw.replace("```json", "").replace("```", "").strip()

## 1️⃣ Extraction de la demande client — *Ollama*

Le premier LLM lit la question du client en langage naturel et en extrait une structure exploitable :
`query`, `company_name`, `company_number`.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """
Tu reçois une demande d'information liée à une entreprise du Luxembourg.
Tu dois en extraire :
- "query" : ce que le client veut savoir (reformulé clairement)
- "company_name" : le nom de l'entreprise s'il est mentionné (sinon null)
- "company_number" : le numéro d'entreprise s'il est mentionné explicitement (sinon null)

Réponds STRICTEMENT en JSON, sans aucun texte autour, avec ce format exact :
{
  "query": "...",
  "company_name": "..." ou null,
  "company_number": "..." ou null
}
"""

@traceable
def extract_request(user_message):
    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    response = ollama_client.chat(model=MODEL_NAME, messages=messages)
    raw = response["message"]["content"]
    try:
        return json.loads(clean_json_fences(raw))
    except json.JSONDecodeError:
        return {"query": user_message, "company_name": None, "company_number": None}

## 2️⃣ Recherche du `company_number` — *OpenCorporates*

Si le numéro d'entreprise n'est pas déjà connu, on interroge le registre officiel du Luxembourg par nom d'entreprise.

In [ ]:
@traceable
def search_comp(company_name):
    url = "https://api.opencorporates.com/v0.4/companies/search"
    params = {
        "q": company_name,
        "jurisdiction_code": "lu",
        "api_token": OPENCORPORATES_TOKEN,
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    companies = data.get("results", {}).get("companies", [])
    return [c["company"]["company_number"] for c in companies]


@traceable
def get_search_comp(extraction):
    """
    Résout le company_number à partir de l'extraction faite par Ollama.
    - Si company_number déjà fourni -> on le garde.
    - Sinon on cherche par company_name et on prend le premier résultat.
    """
    if extraction.get("company_number"):
        return extraction["company_number"], []

    company_name = extraction.get("company_name")
    if not company_name:
        return None, []

    results = search_comp(company_name)
    if not results:
        return None, []

    return results[0], results  # premier choisi + liste complète (utile si ambigu)

## 3️⃣ Vérification B2B — *Google AI Mode (via SerpApi)*

Cette étape n'appelle pas un LLM classique : c'est **Google AI Mode**, interrogé via SerpApi, qui répond directement par OUI/NON. Seules les entreprises B2B (ou partiellement B2B) sont traitées plus loin dans le pipeline.

In [ ]:
@traceable
def call_serpapi(company_number):
    client = serpapi.Client(api_key=SERPAPI_KEY)
    results = client.search({
        "engine": "google_ai_mode",
        "q": (
            f"{company_number} est une entreprise de luxembourg qui est de B2B "
            f"(on tolère si elle est partiellement B2B ou partiellement B2C ou autre) ? "
            f"répondre par juste par OUI ou NON"
        )
    })
    return results["text_blocks"][0]["snippet"]


@traceable
def is_b2b(company_number):
    """Retourne True/False/None (None = réponse ambiguë)."""
    try:
        text = call_serpapi(company_number)
    except Exception:
        return None, ""

    normalized = text.strip().upper()
    has_oui = re.search(r"\bOUI\b", normalized) is not None
    has_non = re.search(r"\bNON\b", normalized) is not None

    if has_oui and not has_non:
        return True, text
    if has_non and not has_oui:
        return False, text
    # ambigu -> par défaut on considère "on ne sait pas", on laisse passer
    return None, text

## 4️⃣ Infos de base + tentative de réponse rapide — *OpenCorporates + Ollama*

En parallèle de la vérification B2B, on récupère les informations officielles de l'entreprise et on tente d'y répondre directement, sans recherche approfondie.

In [ ]:
@traceable
def get_info_comp(company_number):
    url = f"https://api.opencorporates.com/v0.4/companies/lu/{company_number}?api_token={OPENCORPORATES_TOKEN}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    company = data["results"]["company"]
    return {
        "name": company["name"],
        "company_number": company["company_number"],
        "jurisdiction_code": company["jurisdiction_code"],
        "company_type": company["company_type"],
        "incorporation_date": company["incorporation_date"],
        "current_status": company["current_status"],
        "branch_status": company["branch_status"],
        "created_at": company["created_at"],
        "updated_at": company["updated_at"],
    }


BASIC_INFO_SYSTEM_PROMPT = f"""
Tu réponds à une question à partir d'infos de base d'une entreprise (nom, statut,
type, dates d'incorporation, etc.).
- Si ces infos suffisent à répondre complètement à la question, réponds directement.
- Sinon, réponds EXACTEMENT et UNIQUEMENT par : {NOT_FOUND_MARKER}
N'invente rien.
"""


@traceable
def try_answer_from_basic_info(company_info, query):
    if company_info is None:
        return None
    user_prompt = f"Question : {query}\n\nInfos de l'entreprise :\n{json.dumps(company_info, ensure_ascii=False, indent=2)}"
    messages = [
        {"role": "system", "content": BASIC_INFO_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    response = ollama_client.chat(model=MODEL_NAME, messages=messages)
    answer = response["message"]["content"].strip()
    if NOT_FOUND_MARKER in answer:
        return None
    return answer


@traceable
def fetch_info_and_try_answer(company_number, query):
    """Combine récupération infos de base + tentative de réponse (pour le thread parallèle)."""
    try:
        info = get_info_comp(company_number)
    except Exception:
        return None, None
    answer = try_answer_from_basic_info(info, query)
    return info, answer

## 5️⃣ Recherche approfondie dans les statements — *NVIDIA Nemotron*

Si la réponse rapide échoue, l'historique complet de l'entreprise (statements OpenCorporates) est découpé en lots (*chunks*) et confié à un agent NVIDIA qui les examine un à un, en décidant lui-même s'il doit continuer (`continue`) ou conclure (`final`).

> ⚠️ **Limite connue** : les statements OpenCorporates pour le Luxembourg proviennent des publications du Mémorial (registre officiel). Sur les cas testés, les dernières données indexées datent de **2016** — au-delà, l'API ne renvoie rien de plus récent.

In [ ]:
@traceable
def fetch_statements(company_number):
    url = (
        f"https://api.opencorporates.com/v0.4/companies/lu/"
        f"{company_number}/statements?api_token={OPENCORPORATES_TOKEN}"
    )
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    statements = []
    for item in data.get("results", {}).get("statements", []):
        stmt = item["statement"]
        properties = stmt.get("properties", {})
        merged = {
            **properties.get("about", {}),
            **properties.get("home_company", {}),
        }
        statements.append({
            "id": stmt["id"],
            "data_type": stmt["data_type"],
            "properties": clean_html(merged) if merged else properties,
            "start_date": stmt.get("start_date"),
            "date_published": properties.get("date_published"),
        })
    return statements


@traceable
def chunk_list(items, chunk_size):
    for i in range(0, len(items), chunk_size):
        yield items[i:i + chunk_size]


@traceable(run_type="llm")
def call_nvidia_llm_json(system_prompt, user_prompt, max_tokens=2048, reasoning_budget=4096):
    completion = nvidia_client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
        top_p=0.95,
        max_tokens=max_tokens,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": True},
            "reasoning_budget": reasoning_budget,
        },
        stream=False,
    )
    raw = completion.choices[0].message.content.strip()
    return json.loads(clean_json_fences(raw))


AGENT_SYSTEM_PROMPT = """Tu es un agent qui répond à une question en lisant les statements \
d'une entreprise (données OpenCorporates) chunk par chunk, dans l'ordre.

A chaque étape on te donne :
- la question posée,
- ton "état courant" (notes/éléments accumulés depuis les chunks précédents),
- un nouveau chunk de statements,
- un indicateur is_last_chunk (true si c'est le dernier chunk disponible).

Tu dois décider TOI-MEME si tu as besoin de voir d'autres chunks ou non :
- Si la question porte sur un fait ponctuel et que tu as déjà une réponse certaine \
et complète, termine.
- Si la question demande une agrégation/liste, mets à jour ton état avec les \
nouveaux éléments trouvés dans ce chunk, et continue tant qu'il reste des chunks.
- Si is_last_chunk est true, tu DOIS conclure avec "status": "final".
- N'invente jamais d'information absente des statements.

Réponds STRICTEMENT en JSON, sans texte autour :
{
  "status": "final" ou "continue",
  "state": "résumé texte accumulé (utile si status=continue)",
  "answer": "réponse finale complète, uniquement si status=final, sinon null"
}
"""


@traceable
def get_statements_comp(company_number, query, chunk_size=10):
    statements = fetch_statements(company_number)

    if not statements:
        return {"answer": None, "found": False, "message": "Aucun statement trouvé."}

    chunks = list(chunk_list(statements, chunk_size))
    state = ""

    for idx, chunk in enumerate(chunks):
        is_last = (idx == len(chunks) - 1)
        user_prompt = (
            f"Question : {query}\n\n"
            f"État courant : {state or '(vide, premier chunk)'}\n\n"
            f"is_last_chunk : {is_last}\n\n"
            f"Chunk {idx + 1}/{len(chunks)} :\n"
            f"{json.dumps(chunk, ensure_ascii=False, indent=2)}"
        )
        try:
            result = call_nvidia_llm_json(AGENT_SYSTEM_PROMPT, user_prompt)
        except json.JSONDecodeError:
            continue

        if result.get("status") == "final":
            return {
                "answer": result.get("answer"),
                "found": True,
                "chunks_used": idx + 1,
                "total_chunks": len(chunks),
            }
        state = result.get("state", state)

    return {
        "answer": state,
        "found": bool(state),
        "message": "Réponse basée sur le dernier état accumulé (pas de conclusion explicite).",
        "total_chunks": len(chunks),
    }

## 6️⃣ Orchestration de l'agent

La fonction `process_client_request` enchaîne toutes les étapes précédentes :

`Extraction → Recherche entreprise → (Filtre B2B ∥ Réponse rapide) → Recherche approfondie → Réponse finale`

In [ ]:
def process_client_request(user_message):
    # --- Etape 1 : extraction de la demande ---
    extraction = extract_request(user_message)
    query = extraction.get("query") or user_message

    # --- Etape 2 : résolution du company_number ---
    company_number, alternatives = get_search_comp(extraction)
    if not company_number:
        return {
            "status": "error",
            "message": f"Impossible de trouver l'entreprise à partir de la demande : « {user_message} »",
        }

    # --- Etape 3 : appels en parallèle (B2B check + infos de base) ---
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_b2b = executor.submit(is_b2b, company_number)
        future_info = executor.submit(fetch_info_and_try_answer, company_number, query)

        b2b_result, b2b_raw_text = future_b2b.result()
        company_info, quick_answer = future_info.result()

    # --- Etape 4 : si l'entreprise n'est clairement PAS B2B, on arrête ---
    if b2b_result is False:
        return {
            "status": "not_b2b",
            "company_number": company_number,
            "message": "Cette entreprise n'est pas B2B, nous ne pouvons pas traiter cette demande.",
        }

    # --- Etape 5 : si les infos de base suffisent déjà à répondre ---
    if quick_answer:
        return {
            "status": "answered",
            "company_number": company_number,
            "answer": quick_answer,
            "source": "get_info_comp",
        }

    # --- Etape 6 : sinon, recherche approfondie dans les statements ---
    deep_result = get_statements_comp(company_number, query)

    if deep_result.get("found"):
        return {
            "status": "answered",
            "company_number": company_number,
            "answer": deep_result["answer"],
            "source": "get_statements_comp",
            "chunks_used": deep_result.get("chunks_used"),
            "total_chunks": deep_result.get("total_chunks"),
        }

    return {
        "status": "not_found",
        "company_number": company_number,
        "message": deep_result.get("message", "Aucune réponse trouvée."),
    }

## 🧪 7. Test

Exemple d'appel de bout en bout. Chaque étape est visible dans le tableau de bord [LangSmith](https://smith.langchain.com), projet `other_one`.

In [ ]:
result = process_client_request(
    "qui est le directeur de BALKAN FOOD SARL?"
)
print(json.dumps(result, ensure_ascii=False, indent=2))

---
### ✅ Récapitulatif

| Étape | Outil | Type d'appel |
|---|---|---|
| 1. Extraction de la demande | Ollama (`gpt-oss:120b`) | LLM |
| 2. Recherche de l'entreprise | OpenCorporates | API |
| 3. Filtre B2B | Google AI Mode (SerpApi) | Recherche web IA |
| 4. Infos rapides | OpenCorporates + Ollama | API + LLM |
| 5. Analyse approfondie | NVIDIA Nemotron | LLM (raisonnement itératif) |
| 6. Orchestration | — | Logique Python |

*Toutes les clés d'API sont chargées via `google.colab.userdata` — aucune clé n'apparaît en clair dans ce notebook.*